# False Positive Analysis for Claim-Based Contradiction Detection

This notebook investigates **false positives** produced by the `cluster_claims_nli` pipeline:
cases where no contradiction exists in the document (`y_true = 0`) but the model predicted one
with a high confidence score (`p_pred`).

The goal is to understand which claim comparisons are driving the wrong predictions so that
the claim-extraction and NLI steps can be improved.

## 1 — Explore Top False Positives

Load the prediction results and the ContraDoc dataset, then print the 20 false positives
that received the highest contradiction confidence score from the model.

In [4]:
import json
from pathlib import Path

# --- Load data ---
# NLI prediction results produced by the cluster_claims_nli pipeline
results_path = Path("data/results.cluster_claims_nli_r1.json")
# The original ContraDoc dataset (contains ground-truth labels and raw document texts)
dataset_path = Path("datasets/ContraDoc/ContraDoc.json")

results = json.loads(results_path.read_text())
dataset = json.loads(dataset_path.read_text())

# Build a lookup table: unique_id -> example dict, for O(1) access when printing
lookup = {}
for split in ("pos", "neg"):
    for _, ex in dataset[split].items():
        lookup[ex["unique id"]] = ex

# --- Select false positives ---
# False positives are documents without a real contradiction (y_true == 0)
# that the model incorrectly flagged as contradicting.
# Sort by predicted contradiction probability (descending) to surface the worst cases first.
false_positives = [r for r in results if r.get("y_true") == 0 or r.get("y_true") == 0.0]
false_positives.sort(key=lambda x: x.get("p_pred", 0), reverse=True)
selected = false_positives[:20]

print(f"Total false positives found: {len(false_positives)}")
print("Top 20 by predicted contradiction score (p_pred):")
print("=" * 120)

# Print each case with its prediction metadata and the raw document text
for i, r in enumerate(selected, 1):
    uid = r["unique_id"]
    p_pred = r.get("p_pred", 0)
    ex = lookup.get(uid)
    text = ex.get("text", "[text not found]") if ex else "[example not found in dataset]"

    print(
        f"CASE {i:02d} | unique_id={uid} | p_pred={p_pred:.6f} | "
        f"y_true={r.get('y_true')} | doc_type={r.get('doc_type')}"
    )
    print("-" * 120)
    print(text)
    print("=" * 120)

Total false positives found: 413
Top 20 by predicted contradiction score (p_pred):
CASE 01 | unique_id=story_train_8502 | p_pred=0.999512 | y_true=0.0 | doc_type=story
------------------------------------------------------------------------------------------------------------------------
 Opera singer Christine triumphs at the gala on the night of the old managers' retirement. Her old childhood friend, Raoul, hears her sing and recalls his love for Christine. At this time, there are rumors of a phantom living at the Opera and he makes himself known to the managers through letters and malevolent acts. Some time after the gala, the Paris Opera performs Faust, with the prima donna Carlotta playing the lead, against the Phantom's wishes. During the performance, Carlotta loses her voice and the grand chandelier plummets into the audience.
Christine is kidnapped by the phantom and is taken to his home in the cellars of the Opera where he identifies himself as Erik. He plans to keep her there

## 2 — Model Setup and Helper Functions

Load the NLI model (DeBERTa-v3) and the sentence embedding model (MiniLM), then define
helper functions used by the claim-comparison pipeline:

| Function | Purpose |
|---|---|
| `argtopk` | Return indices of the top-k largest values in an array |
| `get_sentence_clusters` | Group each claim with its k most similar neighbours |
| `predict` | Run NLI inference and return per-label probabilities |
| `get_selected_false_positives` | Load and rank false positives from saved results |
| `detect_best_claim_comparison` | Find the claim pair with the highest contradiction score |

In [5]:
import json
import os
from pathlib import Path

import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# --- Configuration ---
NLI_MODEL = "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli"  # NLI model for contradiction detection
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"                  # Sentence encoder used for clustering
TOP_K = 2  # Number of nearest neighbours to include in each claim cluster

# --- Paths ---
results_path = Path("data/results.cluster_claims_nli_r1.json")
dataset_path = Path("datasets/ContraDoc/ContraDoc.json")
claims_dir = Path("extracted_claims")  # One .txt file per document, one claim per line

# Use GPU when available for faster inference
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load the NLI tokenizer and model, move model weights to the target device
nli_tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL)
nli_model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL).to(device)

# Load the sentence embedding model for computing semantic similarity between claims
embed_model = SentenceTransformer(EMBED_MODEL).to(device)


def argtopk(a, k):
    """Return the indices of the top-k largest values in array `a`, sorted descending."""
    idx = np.argpartition(a, -k)[-k:]
    return idx[np.argsort(a[idx])[::-1]]


def get_sentence_clusters(sentences, embeddings, top_k):
    """
    For each sentence, build a cluster containing the sentence itself together
    with its `top_k` most semantically similar neighbours (by cosine similarity).

    Returns a list of clusters, one per input sentence.
    """
    similarity_matrix = cosine_similarity(embeddings)

    # Mask the diagonal so a sentence is never its own nearest neighbour
    for i in range(len(sentences)):
        similarity_matrix[i][i] = -1.0

    cluster_indices = []
    for i in range(len(sentences)):
        cluster_indices.append(argtopk(similarity_matrix[i], top_k))

    clusters = []
    for i, neighbours in enumerate(cluster_indices):
        cluster = [sentences[i], *[sentences[j] for j in neighbours]]
        clusters.append(cluster)

    return clusters


def predict(premise, hypothesis):
    """
    Run NLI inference on a (premise, hypothesis) pair.

    Returns a dict with keys 'entailment', 'neutral', 'contradiction' and
    their corresponding softmax probabilities.
    """
    model_input = nli_tokenizer(premise, hypothesis, truncation=True, return_tensors="pt").to(device)
    output = nli_model(model_input["input_ids"])
    prediction = torch.softmax(output["logits"][0], -1).tolist()
    label_names = ["entailment", "neutral", "contradiction"]
    return {name: float(pred) for pred, name in zip(prediction, label_names)}


def get_selected_false_positives(limit=20):
    """Load prediction results and return the top-`limit` false positives sorted by p_pred."""
    results = json.loads(results_path.read_text())
    false_positives = [r for r in results if r.get("y_true") == 0 or r.get("y_true") == 0.0]
    false_positives.sort(key=lambda x: x.get("p_pred", 0), reverse=True)
    return false_positives[:limit]


def detect_best_claim_comparison(unique_id):
    """
    For the document identified by `unique_id`, find the premise–hypothesis claim pair
    that the NLI model considers most contradictory.

    Searches two types of clusters:
    - Single-claim clusters (each claim tested against each other claim individually)
    - Semantic clusters (each claim tested against its top-k similar neighbours combined)

    Returns a dict with 'cluster', 'premise_claims', 'hypothesis', and 'score',
    or None if no claim file exists or fewer than two claims are available.
    """
    claims_path = claims_dir / f"{unique_id}.txt"
    if not claims_path.exists():
        return None

    with open(claims_path, encoding="utf-8", errors="replace") as f:
        claims = f.readlines()

    # Need at least two claims to form a meaningful premise–hypothesis pair
    if len(claims) <= 1:
        return None

    # Embed all claims and build clusters based on semantic similarity
    embeddings = embed_model.encode(claims)
    single_claims = [[claim] for claim in claims]
    clustered_claims = get_sentence_clusters(claims, embeddings, TOP_K)
    clusters = [*single_claims, *clustered_claims]

    best = None

    # Evaluate every leave-one-out premise–hypothesis split within each cluster
    for cluster in clusters:
        if len(cluster) <= 1:
            continue

        for i in range(len(cluster)):
            # All claims except the current one form the combined premise
            premise_claims = [claim for j, claim in enumerate(cluster) if i != j]
            premise = " ".join(premise_claims)
            hypothesis = cluster[i]
            prediction = predict(premise, hypothesis)
            score = prediction["contradiction"]

            # Track the pair with the highest contradiction score seen so far
            if best is None or score > best["score"]:
                best = {
                    "cluster": cluster,
                    "premise_claims": premise_claims,
                    "hypothesis": hypothesis,
                    "score": score,
                }

    return best

Using device: cuda


Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# Write JSON with currently highest % false positives

In [ ]:
# output_path = Path("data/top20_false_positives_best_claim_comparisons.json")

# # Retrieve the 20 false positives with the highest predicted contradiction scores
# selected = get_selected_false_positives(limit=20)
# export_rows = []

# for case in selected:
#     unique_id = case["unique_id"]

#     # Find the claim comparison that best explains the false positive prediction
#     best = detect_best_claim_comparison(unique_id)

#     # Build the output row; start with metadata from the prediction results
#     row = {
#         "unique_id": unique_id,
#         "doc_type": case.get("doc_type"),
#         "p_pred": case.get("p_pred"),
#         "premise_1": None,
#         "premise_2": None,
#         "hypothesis": None,
#         "contradiction_percent": None,
#     }

#     # Only populate claim fields when a valid comparison with ≥2 premise claims was found
#     if best is not None and len(best["premise_claims"]) >= 2:
#         row.update(
#             {
#                 "premise_1": best["premise_claims"][0].strip(),
#                 "premise_2": best["premise_claims"][1].strip(),
#                 "hypothesis": best["hypothesis"].strip(),
#                 # Convert raw NLI score [0, 1] to a human-readable percentage
#                 "contradiction_percent": round(best["score"] * 100, 2),
#             }
#         )

#     export_rows.append(row)

# # Serialise all rows to a pretty-printed JSON file (non-ASCII characters preserved)
# output_path.write_text(json.dumps(export_rows, indent=2, ensure_ascii=False), encoding="utf-8")
# print(f"Wrote {len(export_rows)} rows to {output_path}")

Wrote 20 rows to data/top20_false_positives_best_claim_comparisons.json


In [7]:
# Add full document text ("story") to each row in the exported false-positive JSON
fp_path = Path("data/top20_false_positives_best_claim_comparisons.json")

rows = json.loads(fp_path.read_text(encoding="utf-8"))
dataset = json.loads(dataset_path.read_text(encoding="utf-8"))

# Build unique_id -> story lookup from ContraDoc
story_lookup = {}
for split in ("pos", "neg"):
    for _, ex in dataset[split].items():
        story_lookup[ex["unique id"]] = ex.get("text")

# Enrich rows with the full story
for row in rows:
    uid = row.get("unique_id")
    row["story"] = story_lookup.get(uid)

fp_path.write_text(json.dumps(rows, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Updated {fp_path} with 'story' for {len(rows)} rows.")

Updated data/top20_false_positives_best_claim_comparisons.json with 'story' for 20 rows.


## 3 — Pairwise Claim Comparison (Alternative Method)

Re-evaluate the 20 false positives using **exhaustive pairwise NLI** on the three claims
already stored per case in the JSON (`premise_1`, `premise_2`, `hypothesis`).

For each case all 6 ordered pairs among those 3 claims are tested, and the pair with the
highest contradiction score is reported. This avoids re-reading the claim files and is
directly comparable to the cluster-based scores written in cell 2.

**Conclusion**: Pair-wise comparison gives kind of the same results - so no need to tweak there.

In [ ]:
# # Load the 20 cases that were written by cell 2
# fp_json = json.loads(
#     Path("data/top20_false_positives_best_claim_comparisons.json").read_text(
#         encoding="utf-8"
#     )
# )

# print("Pairwise contradiction results (from JSON claims) for top-20 false positives")
# print("=" * 120)

# for i, case in enumerate(fp_json, 1):
#     unique_id = case["unique_id"]
#     p_pred = case.get("p_pred", 0)

#     # Collect the (up to 3) claims stored in the JSON for this case
#     claims = [c for c in [case.get("premise_1"), case.get("premise_2"), case.get("hypothesis")] if c]

#     print(
#         f"CASE {i:02d} | unique_id={unique_id} | "
#         f"p_pred={p_pred:.6f} | doc_type={case.get('doc_type')} | "
#         f"cluster_score={case.get('contradiction_percent')}%"
#     )

#     if len(claims) < 2:
#         print("  [fewer than 2 claims available in JSON]")
#         print("-" * 120)
#         continue

#     best = None

#     # Test all ordered pairs (claim_i → premise, claim_j → hypothesis) where i ≠ j
#     for idx_p, premise in enumerate(claims):
#         for idx_h, hypothesis in enumerate(claims):
#             if idx_p == idx_h:
#                 continue

#             prediction = predict(premise, hypothesis)
#             score = prediction["contradiction"]

#             if best is None or score > best["score"]:
#                 best = {"premise": premise, "hypothesis": hypothesis, "score": score}

#     contradiction_pct = round(best["score"] * 100, 2)
#     print(f"  Pairwise contradiction: {contradiction_pct}%")
#     print(f"  Premise   : {best['premise']}")
#     print(f"  Hypothesis: {best['hypothesis']}")
#     print("-" * 120)

Pairwise contradiction results (from JSON claims) for top-20 false positives
CASE 01 | unique_id=story_train_8502 | p_pred=0.999512 | doc_type=story | cluster_score=99.95%
  Pairwise contradiction: 99.95%
  Premise   : Christine promises Erik not to kill herself after becoming his bride.
  Hypothesis: Christine refuses Erik's marriage proposal.
------------------------------------------------------------------------------------------------------------------------
CASE 02 | unique_id=3488771852_4 | p_pred=0.999512 | doc_type=story | cluster_score=99.95%
  Pairwise contradiction: 99.95%
  Premise   : Conan makes short work of his foes in The Scarlet Citadel.
  Hypothesis: Conan is defeated in battle.
------------------------------------------------------------------------------------------------------------------------
CASE 03 | unique_id=3488771834_4 | p_pred=0.999512 | doc_type=story | cluster_score=100.0%
  Pairwise contradiction: 99.95%
  Premise   : Chris Baldry regains his memory.


In [8]:
# ── Phase 1: Re-extract claims for top-20 false positives ──────────────────
import json
from pathlib import Path
from dotenv import dotenv_values
from src.claim_extraction.extractor import extract_claims
from src.claim_extraction.config import DOTENV_PATH

# Load remote model name from .env
_env = dotenv_values(DOTENV_PATH)
_remote_model = (_env.get("CLAIM_MODEL_REMOTE") or "gpt-4o-mini").strip()

# Load top-20 entries (each row has "unique_id" and "story")
_top20_path = Path("data/top20_false_positives_best_claim_comparisons.json")
_top20 = json.loads(_top20_path.read_text(encoding="utf-8"))

_claims_dir = Path("extracted_claims")
_claims_dir.mkdir(parents=True, exist_ok=True)

print(f"Remote model: {_remote_model}")
print(f"Re-extracting claims for {len(_top20)} documents…")
print()

for _i, _row in enumerate(_top20, 1):
    _uid = _row["unique_id"]
    _story = _row.get("story")
    if not _story:
        print(f"  [{_i:02d}] {_uid}: NO STORY TEXT — skipped")
        continue

    print(f"  [{_i:02d}] Extracting claims for {_uid}…")
    _claims = extract_claims(
        text=_story,
        model_name=_remote_model,
        backend="remote",
        use_claimify=False,
        temperature=0.0,
        verbose=True,
    )
    _normalized = [" ".join(str(c).split()) for c in _claims if str(c).strip()]
    (_claims_dir / f"{_uid}.txt").write_text("\n".join(_normalized), encoding="utf-8")
    print(f"         → {len(_normalized)} claims written to extracted_claims/{_uid}.txt")
    print()

print("Done re-extracting claims.")


Remote model: gpt-4o-mini
Re-extracting claims for 20 documents…

  [01] Extracting claims for story_train_8502…
Starting claim extraction | backend=remote | model=gpt-4o-mini | use_claimify=False
Extracted 75 claims.
         → 75 claims written to extracted_claims/story_train_8502.txt

  [02] Extracting claims for 3488771852_4…
Starting claim extraction | backend=remote | model=gpt-4o-mini | use_claimify=False
Extracted 34 claims.
         → 34 claims written to extracted_claims/3488771852_4.txt

  [03] Extracting claims for 3488771834_4…
Starting claim extraction | backend=remote | model=gpt-4o-mini | use_claimify=False
Extracted 52 claims.
         → 52 claims written to extracted_claims/3488771834_4.txt

  [04] Extracting claims for 3488771834_1…
Starting claim extraction | backend=remote | model=gpt-4o-mini | use_claimify=False
Extracted 48 claims.
         → 48 claims written to extracted_claims/3488771834_1.txt

  [05] Extracting claims for story_train_7060…
Starting claim extr

In [9]:
# ── Phase 2: Recompute best-claim contradiction for top-20 ─────────────────
import json
from pathlib import Path

_top20_path = Path("data/top20_false_positives_best_claim_comparisons.json")
_top20 = json.loads(_top20_path.read_text(encoding="utf-8"))

print("Recomputing best-claim contradiction (cluster NLI) for top-20 false positives…")
print()

for _row in _top20:
    _uid = _row["unique_id"]
    _best = detect_best_claim_comparison(_uid)
    _row["premise_1"] = None
    _row["premise_2"] = None
    _row["hypothesis"] = None
    _row["contradiction_percent"] = None

    if _best is not None and len(_best["premise_claims"]) >= 2:
        _row["premise_1"] = _best["premise_claims"][0].strip()
        _row["premise_2"] = _best["premise_claims"][1].strip()
        _row["hypothesis"] = _best["hypothesis"].strip()
        _row["contradiction_percent"] = round(_best["score"] * 100, 2)
        print(f"  {_uid}: {_row['contradiction_percent']}%")
    else:
        print(f"  {_uid}: no usable claims")

_top20_path.write_text(json.dumps(_top20, indent=2, ensure_ascii=False), encoding="utf-8")
print()
print(f"Wrote {len(_top20)} rows to {_top20_path}")


Recomputing best-claim contradiction (cluster NLI) for top-20 false positives…

  story_train_8502: 99.95%
  3488771852_4: 99.95%
  3488771834_4: 100.0%
  3488771834_1: 100.0%
  story_train_7060: 100.0%
  story_train_1324: 100.0%
  3488771834_3: 100.0%
  story_train_6589: 99.95%
  story_train_1147: 100.0%
  story_train_8730: 99.95%
  story_train_6268: 99.95%
  story_train_8644: 100.0%
  wiki_train_29379: 99.95%
  story_train_4565: 100.0%
  3488771852_1: 99.95%
  story_train_8006: 99.95%
  story_train_9500: 99.95%
  wiki_train_29049: 99.95%
  story_train_1720: 99.95%
  wiki_train_29316: 99.95%

Wrote 20 rows to data/top20_false_positives_best_claim_comparisons.json


In [10]:
# ── Phase 3: Print pairwise contradiction results for top-20 ───────────────
import json
from pathlib import Path

_fp_json = json.loads(
    Path("data/top20_false_positives_best_claim_comparisons.json").read_text(encoding="utf-8")
)

print("Pairwise contradiction results (from JSON claims) for top-20 false positives")
print("=" * 120)

for _i, _case in enumerate(_fp_json, 1):
    _uid = _case["unique_id"]
    _p_pred = _case.get("p_pred", 0)
    _claims = [c for c in [_case.get("premise_1"), _case.get("premise_2"), _case.get("hypothesis")] if c]

    print(
        f"CASE {_i:02d} | unique_id={_uid} | "
        f"p_pred={_p_pred:.6f} | doc_type={_case.get('doc_type')} | "
        f"cluster_score={_case.get('contradiction_percent')}%"
    )

    if len(_claims) < 2:
        print("  [fewer than 2 claims available in JSON]")
        print("-" * 120)
        continue

    _best = None
    for _idx_p, _premise in enumerate(_claims):
        for _idx_h, _hypothesis in enumerate(_claims):
            if _idx_p == _idx_h:
                continue
            _prediction = predict(_premise, _hypothesis)
            _score = _prediction["contradiction"]
            if _best is None or _score > _best["score"]:
                _best = {"premise": _premise, "hypothesis": _hypothesis, "score": _score}

    _contradiction_pct = round(_best["score"] * 100, 2)
    print(f"  Pairwise contradiction: {_contradiction_pct}%")
    print(f"  Premise   : {_best['premise']}")
    print(f"  Hypothesis: {_best['hypothesis']}")
    print("-" * 120)


Pairwise contradiction results (from JSON claims) for top-20 false positives
CASE 01 | unique_id=story_train_8502 | p_pred=0.999512 | doc_type=story | cluster_score=99.95%
  Pairwise contradiction: 99.95%
  Premise   : Erik has trapped the Persian in a hot torture chamber.
  Hypothesis: Erik allows the Persian to escape.
------------------------------------------------------------------------------------------------------------------------
CASE 02 | unique_id=3488771852_4 | p_pred=0.999512 | doc_type=story | cluster_score=99.95%
  Pairwise contradiction: 99.95%
  Premise   : Conan makes short work of his foes in "The Scarlet Citadel".
  Hypothesis: Conan is defeated in battle.
------------------------------------------------------------------------------------------------------------------------
CASE 03 | unique_id=3488771834_4 | p_pred=0.999512 | doc_type=story | cluster_score=100.0%
  Pairwise contradiction: 99.95%
  Premise   : Chris Baldry regains his memory.
  Hypothesis: Chris Ba